# OCSD – Chạy toàn bộ thực nghiệm (Run all)

Khóa luận: *Sinh ảnh cảnh nhất quán đối tượng từ phác thảo và văn bản bằng mô hình khuếch tán* – Lê Hoàng Bảo (21090331), Thái Bá Cường (21050681); GVHD: TS. Nguyễn Thanh Chuyên.

**Cách dùng:** Runtime → Change runtime type → GPU (L4 hoặc A100) → sửa ô 1 → Runtime → Run all.

Notebook chạy 4 giai đoạn: (A) dựng bộ dữ liệu, (B) sinh ảnh cho E1–E4 và khảo sát α, (C) đánh giá, (D) xuất bảng/hình/tóm tắt. Mọi kết quả nằm trong `DRIVE_ROOT/results` (`summary.md`, `summary.json`, `tables/`, `figures/`, `progress.json`). Nếu Colab ngắt kết nối, chỉ cần Run all lại: phần đã xong được bỏ qua.

In [ ]:
#@title 1. Cấu hình (chỉ sửa ô này)
REPO_URL   = "https://github.com/bobbibao/ocsd-thesis.git"  #@param {type:"string"}
BRANCH     = "main"                                             #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"                 #@param {type:"string"}
TIER       = "pilot"   #@param ["pilot", "paper", "full"]
BACKBONE   = "sd15"    #@param ["sd15", "sd21"]
MAX_MINUTES = 600      #@param {type:"number"}
HF_TOKEN   = ""        #@param {type:"string"}
# pilot: kiểm tra nhanh (~1-2 giờ GPU); paper: đủ số liệu cho khóa luận; full: cho bài báo (cần thêm GPU).
# Có thể bấm "Run all" nhiều lần: ảnh/độ đo đã có trên Drive sẽ được bỏ qua và chạy tiếp phần còn lại.

In [ ]:
#@title 2. Gắn Google Drive, kiểm tra GPU, lấy mã nguồn mới nhất
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
CODE = "/content/KLTN_OCSD"
if not REPO_URL:
    # chưa có repo: dùng bản mã nguồn đã tải lên Drive (thư mục DRIVE_ROOT/code)
    CODE = os.path.join(DRIVE_ROOT, "code")
elif os.path.exists(CODE):
    subprocess.run(f"git -C {CODE} fetch -q origin {BRANCH} && git -C {CODE} reset -q --hard origin/{BRANCH}", shell=True, check=True)
else:
    subprocess.run(f"git clone -q -b {BRANCH} {REPO_URL} {CODE}", shell=True, check=True)
print("Mã nguồn:", CODE, subprocess.run(f"git -C {CODE} log -1 --format='%h %s' 2>/dev/null", shell=True, capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
os.environ["HF_HOME"] = os.path.join(DRIVE_ROOT, "hf_cache")   # lưu mô hình trên Drive: lần sau không tải lại
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
#@title 3. Cài thư viện (giữ nguyên PyTorch của Colab)
!pip install -q -r {CODE}/requirements-colab.txt
import importlib, ocsd; importlib.reload(ocsd)
from ocsd.config import Paths, ExperimentConfig, TIERS
P = Paths(DRIVE_ROOT); P.makedirs()
E = ExperimentConfig(backbone=BACKBONE, tier=TIER)
E.save(os.path.join(P.results, "experiment_config.json"))
print(E)

## A. Bộ dữ liệu
QuickDraw-Scenes (tổng hợp, 12 ô: số đối tượng 1/3/5/8+ × độ phức tạp đơn giản/trung bình/phức tạp) và COCO-Sketch (ảnh thật COCO val2017, phác thảo PiDiNet, mặt nạ thật). Lần đầu mất ~10–20 phút (tải COCO 1 GB).

In [ ]:
#@title A. Dựng bộ dữ liệu (bỏ qua nếu đã có)
from ocsd import stages
stages.setup_data(P, TIER, with_coco=True)
from IPython.display import Image, display
display(Image(os.path.join(P.results, "figures", "benchmark_examples.png"), width=900))

## B. Sinh ảnh
Thứ tự: E3 QuickDraw (mọi baseline + OCSD-lite trên toàn bộ tập) → OCSD và Zhang et al. (cần học định danh, trên tập con phân tầng) → E3 COCO → E4 cắt bỏ → khảo sát α. `MAX_MINUTES` giới hạn thời gian của ô này; chạy lại để tiếp tục.

In [ ]:
#@title B1. Kế hoạch và số ảnh còn lại
plan = stages.experiment_plan(P, E)
for j in plan:
    print(f"{j['name']:22s} {len(j['scenes']):4d} cảnh x {len(j['methods']):2d} phương pháp x {len(j['seeds'])} seed = còn {stages._count_todo(P, j)} ảnh")

In [ ]:
#@title B2. Sinh ảnh (có thể chạy lại để tiếp tục)
import time, torch
t0 = time.time()
eng, vis = stages.generate(P, E, max_minutes=MAX_MINUTES)
print(f"Xong phiên sinh ảnh sau {(time.time()-t0)/60:.1f} phút")

## C. Đánh giá
OPR, OCE, độ chính xác đếm, mIoU, RA bằng bộ phát hiện **OWLv2** (khác Grounding DINO mà phương pháp dùng bên trong, để tránh tối ưu hóa theo chính độ đo); CLIP score toàn ảnh và cấp đối tượng; ID-Sim (DINOv2); FID/KID; LPIPS.

In [ ]:
#@title C. Tính độ đo
del eng; torch.cuda.empty_cache()
vis = stages.evaluate(P, E, vis=vis, fid=True)

## D. Báo cáo

In [ ]:
#@title D. Bảng, hình, kiểm định thống kê, tóm tắt
summary = stages.report(P, E)
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(P.results, "summary.md")).read()))
for f in summary["figures"]:
    display(Image(f, width=900))